# 13. Target and frequency encoding on every column

One variable against ledger row 9 (`lgbm_bag08_seed42`, CV 0.963471). Identical
model, identical folds, identical seed. The only change is the feature set:
twelve raw columns become twelve raw columns plus a smoothed target mean and a
level frequency for each.

The idea is not ours. A public notebook by tomasa2 measured +0.0023 CV from it and
reported it as larger than every model choice, tuning decision and ensembling trick
in that notebook combined. Our repo spent the whole competition on models and never
touched the representation, so this is the first honest test of that claim here.

**Why it is not absurd to target-encode a continuous column.**
`daily_screen_time_hours` has about 1,380 distinct values over 691,369 rows, so a
level holds ~500 rows and its smoothed target mean is a well-estimated quantity at
1,380 points. A tree spends a few dozen splits approximating the same curve.

**Leakage is the entire difficulty**, so it is checked by execution below rather
than argued. Three checks run before any training.


In [ ]:
# Set False for the real run. Smoke mode exercises every line on 20k rows.
SMOKE = True

SEED = 42
N_INNER = 5
SMOOTH = 10.0

# Ledger row 9: the same config with the raw feature set. This is the number the
# experiment is measured against, and the comparison is paired per fold.
BASELINE_NAME = "lgbm_bag08_seed42"
BASELINE_CV = 0.963471
EXPECTED_FOLD_SHA = "ec282b0968059676"

PARAMS = dict(
    objective="binary", metric="auc", learning_rate=0.05, n_estimators=2000,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    random_state=SEED, n_jobs=-1, verbose=-1,
    deterministic=True, force_row_wise=True,
)
print(f"SMOKE = {SMOKE}")


In [ ]:
import hashlib
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold


def locate(name):
    kag = Path("/kaggle/input")
    if kag.exists():
        hits = sorted(kag.rglob(name))
        if hits:
            return hits[0]
    for b in [Path.cwd(), *Path.cwd().parents]:
        p = b / "data" / "raw" / name
        if p.exists():
            return p
    raise FileNotFoundError(name)


train = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
TARGET = "addicted_label"
CAT = ["gender", "stress_level", "academic_work_impact"]
COLS = [c for c in train.columns if c not in ("id", TARGET)]

if SMOKE:
    train = train.sample(20000, random_state=0).reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
    PARAMS["n_estimators"] = 200

y = train[TARGET].to_numpy()
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

sha = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
ALIGNED = sha == EXPECTED_FOLD_SHA
print(f"rows {len(train)}   target rate {y.mean():.6f}")
print(f"fold sizes {np.bincount(folds).tolist()}")
print(f"fold sha {sha}  expected {EXPECTED_FOLD_SHA}")
if SMOKE:
    print("SMOKE: subsampled, so the sha is EXPECTED to differ. Not a check.")
else:
    print("fold alignment: VERIFIED" if ALIGNED else
          "fold alignment: MISMATCH - the OOF from this run is not blendable")

X = train[COLS].copy()
X_test = test[COLS].copy()
for c in CAT:
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")


## The encoder

A row's encoding is **not a property of the row**. The same row has one encoding
when it is the validation data and a different one when it is another fold's
training data, so the design matrix is rebuilt inside the fold loop rather than
computed once. A first draft of this stored one encoded column per feature and let
each outer fold overwrite it, which quietly handed the model encodings built from
the rows it was about to be scored on.

The inner split is a plain `KFold`, not stratified. Stratifying would make the
split a function of the target, which buys nothing at this size and would make the
leak check below untestable: flipping labels would reshuffle the blocks, so any
before/after comparison would be across different partitions.


In [ ]:
def _stats(levels, yy, prior):
    """Smoothed target mean and level frequency, fit only on the rows given."""
    df = pd.DataFrame({"v": levels, "y": yy})
    g = df.groupby("v", dropna=False, observed=True)["y"].agg(["sum", "count"])
    mean = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)
    return mean, g["count"] / len(df)


def _apply(levels, mean, freq, prior):
    s = pd.Series(levels)
    m = s.map(mean).to_numpy(dtype=np.float64)
    f = s.map(freq).to_numpy(dtype=np.float64)
    # A level unseen while fitting falls back to the prior and zero mass.
    return np.nan_to_num(m, nan=prior), np.nan_to_num(f, nan=0.0)


def encode_fold(Xf, yy, tr, va, Xt=None, seed=SEED):
    """Encodings for ONE outer fold: (train, valid, test)."""
    prior = float(yy[tr].mean())
    e_tr, e_va, e_te = {}, {}, {}
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(tr))

    for c in COLS:
        lv = Xf[c].to_numpy()
        # Fit once on the whole training portion, for validation and test rows.
        mean, freq = _stats(lv[tr], yy[tr], prior)
        e_va[f"te_{c}"], e_va[f"fq_{c}"] = _apply(lv[va], mean, freq, prior)
        if Xt is not None:
            e_te[f"te_{c}"], e_te[f"fq_{c}"] = _apply(Xt[c].to_numpy(), mean,
                                                      freq, prior)
        # Training rows get inner out-of-fold values.
        tm, tf = np.empty(len(tr)), np.empty(len(tr))
        for itr, iva in splits:
            im, if_ = _stats(lv[tr[itr]], yy[tr[itr]], prior)
            tm[iva], tf[iva] = _apply(lv[tr[iva]], im, if_, prior)
        e_tr[f"te_{c}"], e_tr[f"fq_{c}"] = tm, tf

    return (pd.DataFrame(e_tr), pd.DataFrame(e_va),
            pd.DataFrame(e_te) if Xt is not None else None)


def build(Xf, yy, tr, va, Xt=None, seed=SEED):
    d_tr, d_va, d_te = encode_fold(Xf, yy, tr, va, Xt, seed)
    Xtr = pd.concat([Xf.iloc[tr].reset_index(drop=True), d_tr], axis=1)
    Xva = pd.concat([Xf.iloc[va].reset_index(drop=True), d_va], axis=1)
    Xte = None if Xt is None else pd.concat(
        [Xt.reset_index(drop=True), d_te], axis=1)
    return Xtr, Xva, Xte


print(f"{len(COLS)} raw columns -> {len(COLS) * 3} features after encoding")


## Leak checks, by execution

Printed rather than asserted, so a failure still leaves the notebook's outputs
readable. Read all three together: the first two must be ~0, the third must be
large. Without the third, a broken encoder that ignored the target entirely would
pass the first two and look clean.


In [ ]:
_tr = np.where(folds != 0)[0]
_va = np.where(folds == 0)[0]
d_tr0, d_va0, _ = encode_fold(X, y, _tr, _va)

# 1. A validation row's own target must never reach its own encoding.
y1 = y.copy()
y1[_va] = 1 - y1[_va]
_, d_va1, _ = encode_fold(X, y1, _tr, _va)
leak1 = max(np.abs(d_va0[f"te_{c}"] - d_va1[f"te_{c}"]).max() for c in COLS)

# 2. A training row's own target must never reach its own inner encoding. A small
# residual is expected and is not a leak: `prior` is the training-portion mean, so
# flipping any row moves it, and the prior enters every smoothed mean. Flip few
# enough rows that the residual is visibly prior-sized rather than label-sized.
_, iva0 = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(_tr))[0]
pick = iva0[:200]
y2 = y.copy()
y2[_tr[pick]] = 1 - y2[_tr[pick]]
d_tr2, _, _ = encode_fold(X, y2, _tr, _va)
leak2 = max(np.abs(d_tr0[f"te_{c}"].to_numpy()[pick]
                   - d_tr2[f"te_{c}"].to_numpy()[pick]).max() for c in COLS)
prior_shift = abs(float(y2[_tr].mean()) - float(y[_tr].mean()))

# 3. The encoding MUST move when targets it is allowed to see change.
y3 = y.copy()
y3[_tr] = 1 - y3[_tr]
_, d_va3, _ = encode_fold(X, y3, _tr, _va)
live = max(np.abs(d_va0[f"te_{c}"] - d_va3[f"te_{c}"]).max() for c in COLS)

print(f"1. flip all validation targets -> change in their encoding: {leak1:.3e}")
print(f"   must be 0. Their rows are not in the fitting set at all.")
print(f"2. flip 200 training rows -> change in their own encoding:  {leak2:.3e}")
print(f"   prior moved {prior_shift:.3e}. These should match: a level absent from")
print(f"   the fitting rows encodes to the prior exactly, so it tracks it.")
print(f"3. flip all training targets -> change in val encoding:     {live:.3e}")
print(f"   must be LARGE, or the encoder is not using the target and 1 and 2")
print(f"   are passing vacuously.")

CLEAN = leak1 == 0 and leak2 < 10 * max(prior_shift, 1e-9) and live > 0.1
print()
print("LEAK CHECKS: PASS" if CLEAN else "LEAK CHECKS: FAILED - do not log this run")


In [ ]:
oof = np.zeros(len(train))
test_pred = np.zeros(len(test))
scores = []
t0 = time.time()

for f in range(5):
    tr = np.where(folds != f)[0]
    va = np.where(folds == f)[0]
    Xtr, Xva, Xte = build(X, y, tr, va, X_test)

    m = lgb.LGBMClassifier(**PARAMS)
    m.fit(Xtr, y[tr])
    p = m.predict_proba(Xva)[:, 1]
    oof[va] = p
    test_pred += m.predict_proba(Xte)[:, 1] / 5
    scores.append(roc_auc_score(y[va], p))
    print(f"  fold {f}: {scores[-1]:.6f}   [{(time.time() - t0) / 60:.1f} min]")

cv = float(np.mean(scores))
sd = float(np.std(scores))
print()
print(f"CV {cv:.6f} +/- {sd:.6f}")
print(f"OOF AUC (all rows at once) {roc_auc_score(y, oof):.6f}")


In [ ]:
# Paired against ledger row 9. The right bar for "is B better than A" is the
# spread of the per-fold DIFFERENCES and how many folds it wins, not the fold
# spread, which is common to both models and cancels. See NOTES.md.
base_path = None
for b in [Path.cwd(), *Path.cwd().parents]:
    p = b / "artifacts" / "oof" / "lgbm_bag08_lr005_n2000_seed42.npy"
    if p.exists():
        base_path = p
        break

print(f"CV this run {cv:.6f}   ledger row 9 {BASELINE_CV:.6f}"
      f"   diff {cv - BASELINE_CV:+.6f}")

if base_path is not None and not SMOKE:
    base = np.load(base_path)
    bf = np.array([roc_auc_score(y[folds == f], base[folds == f]) for f in range(5)])
    d = np.array(scores) - bf
    print(f"baseline OOF reproduces its ledger row to {bf.mean() - BASELINE_CV:+.2e}")
    print(f"per-fold differences: {np.round(d, 6).tolist()}")
    print(f"mean {d.mean():+.6f}, paired sd {d.std(ddof=1):.6f}, "
          f"wins {(d > 0).sum()}/5 folds")
    # CLAUDE.md: a jump over ~2% is treated as a leak until proven otherwise.
    print(f"relative change {(cv - BASELINE_CV) / BASELINE_CV:+.4%} "
          f"(>2% would mean stop and investigate)")
else:
    print("paired comparison skipped: smoke run, or baseline OOF not found")


In [ ]:
prefix = "SMOKE_" if SMOKE else ""
out = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()

np.save(out / f"{prefix}te_oof.npy", oof)
np.save(out / f"{prefix}te_test.npy", test_pred)
pd.DataFrame({"id": test["id"], TARGET: test_pred}).to_csv(
    out / f"{prefix}submission.csv", index=False)

print(f"wrote {prefix}te_oof.npy, {prefix}te_test.npy, {prefix}submission.csv")
print()
print("ledger line:")
print(f"  name    lgbm_bag08_seed42_te")
print(f"  cv_mean {cv:.6f}")
print(f"  cv_std  {sd:.6f}")
print(f"  leak checks {'PASS' if CLEAN else 'FAILED'}, "
      f"fold alignment {'verified' if ALIGNED else 'NOT verified'}")
